# NexTek AI Agent
### Agente de IA para consulta de documentos internos
**Stack:** Python - LangChain - Google Gemini - PyPDF

**Flujo:**
```
PDFs -> PyPDF -> LangChain -> FAISS -> Gemini -> Usuario
```

## PASO 1 - Instalacion de dependencias

In [ ]:
!pip install -q langchain langchain-google-genai langchain-community langchain-text-splitters pypdf faiss-cpu google-generativeai
print('Dependencias instaladas correctamente')

## PASO 2 - Configurar API Key de Google Gemini
Obtener key gratuita en: https://aistudio.google.com/app/apikey

IMPORTANTE: La key debe ir entre comillas simples, ejemplo:
GOOGLE_API_KEY = 'AIzaSyXXXXXXXXXXXXXXXXXXXXXX'

In [ ]:
import os

GOOGLE_API_KEY = 'TU_API_KEY_AQUI'

os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY

if GOOGLE_API_KEY == 'TU_API_KEY_AQUI':
    print('AVISO: Reemplaza TU_API_KEY_AQUI con tu key real entre comillas simples')
else:
    print('API Key configurada correctamente')

## PASO 3 - Subir los 5 PDFs de NexTek
Al ejecutar esta celda aparece un boton para seleccionar archivos.

In [ ]:
import os
from google.colab import files

os.makedirs('/content/docs', exist_ok=True)

print('Selecciona los 5 archivos PDF de NexTek...')
uploaded = files.upload()

for filename in uploaded.keys():
    os.rename(filename, f'/content/docs/{filename}')
    print(f'OK: {filename}')

pdfs = [f for f in os.listdir('/content/docs') if f.endswith('.pdf')]
print(f'Total PDFs cargados: {len(pdfs)}')

## PASO 4 - Construir la base de conocimiento
PyPDF extrae el texto, LangChain divide en chunks, FAISS crea el indice vectorial.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

print('Cargando documentos PDF...')
docs_dir = '/content/docs'
all_documents = []

pdf_files = sorted([f for f in os.listdir(docs_dir) if f.endswith('.pdf')])

for pdf_file in pdf_files:
    path = os.path.join(docs_dir, pdf_file)
    loader = PyPDFLoader(path)
    pages = loader.load()
    all_documents.extend(pages)
    print(f'OK: {pdf_file} - {len(pages)} paginas')

print(f'Total paginas cargadas: {len(all_documents)}')

print('Dividiendo texto en fragmentos...')
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=['\n\n', '\n', '. ', ' ', '']
)
chunks = splitter.split_documents(all_documents)
print(f'Total fragmentos generados: {len(chunks)}')

print('Generando embeddings con Google Gemini...')
embeddings = GoogleGenerativeAIEmbeddings(
    model='models/embedding-001',
    google_api_key=GOOGLE_API_KEY
)

print('Construyendo indice FAISS (puede tardar 1-2 minutos)...')
vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local('/content/nextek_index')
print('Base de conocimiento lista y guardada')

## PASO 5 - Crear el Agente NexTek

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

llm = ChatGoogleGenerativeAI(
    model='gemini-1.5-flash',
    google_api_key=GOOGLE_API_KEY,
    temperature=0.2,
    convert_system_message_to_human=True
)

retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

PROMPT_TEMPLATE = """Eres el asistente virtual oficial de NexTek, una tienda de comercio electronico especializada en electronica y tecnologia en Mexico.

Tu funcion es responder preguntas sobre los documentos internos de NexTek: politica de privacidad, politica de reembolsos y devoluciones, preguntas frecuentes, guia de envios y terminos y condiciones.

REGLAS:
- Responde SIEMPRE en español, de forma clara y amigable.
- Basa tu respuesta UNICAMENTE en el contexto proporcionado.
- Si la informacion no esta en los documentos responde: Esa informacion no se encuentra en los documentos de NexTek.
- Usa listas cuando la respuesta tenga varios puntos.
- No inventes informacion.

CONTEXTO:
{context}

PREGUNTA:
{question}

RESPUESTA:"""

prompt = PromptTemplate(
    template=PROMPT_TEMPLATE,
    input_variables=['context', 'question']
)

agente_nextek = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type='stuff',
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={'prompt': prompt}
)

print('Agente NexTek listo para recibir preguntas')

## PASO 6 - Probar el agente con preguntas de ejemplo

In [ ]:
def preguntar(pregunta, mostrar_fuentes=False):
    print('\n' + '='*60)
    print(f'PREGUNTA: {pregunta}')
    print('-'*60)
    resultado = agente_nextek.invoke({'query': pregunta})
    respuesta = resultado['result']
    print(f'RESPUESTA:\n{respuesta}')
    if mostrar_fuentes and resultado.get('source_documents'):
        print('\nFUENTES:')
        fuentes_vistas = set()
        for doc in resultado['source_documents']:
            fuente = doc.metadata.get('source', 'Desconocida').split('/')[-1]
            pagina = doc.metadata.get('page', '?')
            clave = f'{fuente} - pag. {pagina}'
            if clave not in fuentes_vistas:
                print(f'  - {clave}')
                fuentes_vistas.add(clave)
    return respuesta

preguntas_ejemplo = [
    'Cuanto tiempo tengo para devolver un smartphone?',
    'El envio es gratis en NexTek?',
    'Como puedo ejercer mis derechos ARCO?',
    'Que metodos de pago aceptan?',
    'Que cubre la garantia NexTek?'
]

for pregunta in preguntas_ejemplo:
    preguntar(pregunta, mostrar_fuentes=True)

print('\n' + '='*60)
print('Pruebas completadas exitosamente')

## PASO 7 - Modo conversacion interactivo
Escribe tus propias preguntas. Escribe 'salir' para terminar.

In [ ]:
print('Agente NexTek activo. Escribe tu pregunta o salir para terminar.\n')

while True:
    pregunta = input('Tu pregunta: ').strip()
    if not pregunta:
        continue
    if pregunta.lower() in ['salir', 'exit', 'quit']:
        print('Agente NexTek desconectado.')
        break
    preguntar(pregunta, mostrar_fuentes=True)

## PASO 8 - Cargar el indice en sesiones futuras
Ejecuta esta celda si ya tienes el indice guardado y quieres reutilizarlo sin reprocesar los PDFs.

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model='models/embedding-001',
    google_api_key=GOOGLE_API_KEY
)

vectorstore = FAISS.load_local(
    '/content/nextek_index',
    embeddings,
    allow_dangerous_deserialization=True
)

print('Indice FAISS cargado correctamente')
print(f'Fragmentos disponibles: {vectorstore.index.ntotal}')